# 13장 저장, 추론, 하이퍼파라미터 — 실행을 기록으로 남기기

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch13_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `train.py` 와 `infer.py` 를 치고, 기본값·`--lr 0.05 --epochs 3 --run lr005`·`--help` 세 번 실행. `results/` 폴더 열어 보기 | [ch13_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch13_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | `train.py` 에서 12장과 달라진 줄에만 주석을 달기(무엇을 위해 추가됐는지) | [ch13_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch13_2_comment.ipynb) |
| 3. 다시 만들기 | 12장 `train_mnist.py` 를 열고, 이 장을 보지 않고 세 가지(argparse, 결과 폴더, best/last 저장)를 직접 추가. 그다음 `infer.py` 를 빈 파일에서 작성 | [ch13_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch13_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "13"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/13")

준비 끝. 작업 폴더: code/13


## `train.py` — 13장 train.py: 12장 학습 코드에 (1) 명령줄 인자 (2) 결과 폴더와 설정 저장 (3) 가중치 저장 (4) 지표 기록을 더한 것.

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 13장 train.py: 12장 학습 코드에 (1) 명령줄 인자 (2) 결과 폴더와 설정 저장 (3) 가중치 저장 (4) 지표 기록을 더한 것.
# 실행 예:
#   python train.py                                  # 기본값으로
#   python train.py --epochs 3 --lr 0.05 --run lr005  # 인자를 바꿔서
#   python train.py --help                           # 인자 목록
import argparse
import json
import os
import time

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


def parse_args():
    p = argparse.ArgumentParser(description="MNIST MLP 학습")
    p.add_argument("--data-root", default=os.environ.get("JUCOS_DATA", "./data"), help="데이터 폴더")
    p.add_argument("--results", default="results", help="결과를 저장할 상위 폴더")
    p.add_argument("--run", default=None, help="이 실행의 이름 (없으면 시각으로)")
    p.add_argument("--epochs", type=int, default=2)
    p.add_argument("--batch-size", type=int, default=64)
    p.add_argument("--hidden", type=int, default=128)
    p.add_argument("--lr", type=float, default=0.1)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--log-every", type=int, default=200)
    return p.parse_args()


class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=128, out_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x.flatten(1))))


def evaluate(model, loader, loss_fn, device):
    """정확도와 평균 손실을 함께 돌려준다."""
    model.eval()
    correct, total_loss = 0, 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            scores = model(images)
            total_loss += loss_fn(scores, labels).item() * len(labels)   # 배치 평균 × 개수 = 합
            correct += (scores.argmax(1) == labels).sum().item()
    model.train()
    n = len(loader.dataset)
    return correct / n, total_loss / n


def main():
    args = parse_args()
    torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 결과 폴더: results/{run}/ 에 이 실행의 모든 것을 모은다
    run_name = args.run or time.strftime("%Y%m%d-%H%M%S")
    out_dir = os.path.join(args.results, run_name)
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(vars(args), f, ensure_ascii=False, indent=2)          # 4장의 JSON: 설정을 그대로 남긴다

    transform = transforms.ToTensor()
    train_set = datasets.MNIST(args.data_root, train=True, download=True, transform=transform)
    test_set = datasets.MNIST(args.data_root, train=False, download=True, transform=transform)
    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=1000)

    model = MLP(hidden=args.hidden).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)

    history = []                                                         # 에포크별 지표를 모아 둔다
    best_acc = 0.0
    for epoch in range(1, args.epochs + 1):
        running, t0 = 0.0, time.time()
        for step, (images, labels) in enumerate(train_loader, start=1):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
            running += loss.item()
            if step % args.log_every == 0:
                print(f"  에포크 {epoch} 걸음 {step:4d}  평균 손실 {running / args.log_every:.4f}")
                running = 0.0
        acc, test_loss = evaluate(model, test_loader, loss_fn, device)
        history.append({"epoch": epoch, "test_acc": acc, "test_loss": test_loss, "seconds": time.time() - t0})
        print(f"에포크 {epoch}: 시험 정확도 {acc:.4f}  시험 손실 {test_loss:.4f}")
        if acc > best_acc:                                               # 가장 좋은 에포크의 가중치만 남긴다
            best_acc = acc
            torch.save(model.state_dict(), os.path.join(out_dir, "model_best.pt"))
    torch.save(model.state_dict(), os.path.join(out_dir, "model_last.pt"))
    with open(os.path.join(out_dir, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump({"best_test_acc": best_acc, "history": history}, f, ensure_ascii=False, indent=2)
    print(f"저장 완료: {out_dir}/  (config.json, model_best.pt, model_last.pt, metrics.json)")


if __name__ == "__main__":
    main()
```

책의 첫 명령은 이름 없이 `python train.py` 입니다(실행 이름이 시각으로 붙습니다). 여기서는 26장에서 이 모델을 다시 쓰려고 이름을 `lr01` 로 줍니다.

In [ ]:
!python train.py --run lr01

  에포크 1 걸음  200  평균 손실 0.8553
  에포크 1 걸음  400  평균 손실 0.3859
  에포크 1 걸음  600  평균 손실 0.3291
  에포크 1 걸음  800  평균 손실 0.2942
에포크 1: 시험 정확도 0.9250  시험 손실 0.2607
  에포크 2 걸음  200  평균 손실 0.2533
  에포크 2 걸음  400  평균 손실 0.2304
  에포크 2 걸음  600  평균 손실 0.2312
  에포크 2 걸음  800  평균 손실 0.2108
에포크 2: 시험 정확도 0.9473  시험 손실 0.1804
저장 완료: results\lr01/  (config.json, model_best.pt, model_last.pt, metrics.json)


In [ ]:
!python train.py --lr 0.05 --epochs 3 --run lr005

  에포크 1 걸음  200  평균 손실 1.1927
  에포크 1 걸음  400  평균 손실 0.4879
  에포크 1 걸음  600  평균 손실 0.3938
  에포크 1 걸음  800  평균 손실 0.3501
에포크 1: 시험 정확도 0.9114  시험 손실 0.3125
  에포크 2 걸음  200  평균 손실 0.3148
  에포크 2 걸음  400  평균 손실 0.2942
  에포크 2 걸음  600  평균 손실 0.3010
  에포크 2 걸음  800  평균 손실 0.2790
에포크 2: 시험 정확도 0.9300  시험 손실 0.2493
  에포크 3 걸음  200  평균 손실 0.2524
  에포크 3 걸음  400  평균 손실 0.2432
  에포크 3 걸음  600  평균 손실 0.2392
  에포크 3 걸음  800  평균 손실 0.2403
에포크 3: 시험 정확도 0.9382  시험 손실 0.2150
저장 완료: results\lr005/  (config.json, model_best.pt, model_last.pt, metrics.json)


In [ ]:
!python train.py --help

usage: train.py [-h] [--data-root DATA_ROOT] [--results RESULTS] [--run RUN]
                [--epochs EPOCHS] [--batch-size BATCH_SIZE] [--hidden HIDDEN]
                [--lr LR] [--seed SEED] [--log-every LOG_EVERY]

MNIST MLP 학습

options:
  -h, --help            show this help message and exit
  --data-root DATA_ROOT
                        데이터 폴더
  --results RESULTS     결과를 저장할 상위 폴더
  --run RUN             이 실행의 이름 (없으면 시각으로)
  --epochs EPOCHS
  --batch-size BATCH_SIZE
  --hidden HIDDEN
  --lr LR
  --seed SEED
  --log-every LOG_EVERY


## 결과 폴더

In [ ]:
import os
print(sorted(os.listdir('results/lr005')))

['config.json', 'metrics.json', 'model_best.pt', 'model_last.pt']


In [ ]:
print(open('results/lr005/metrics.json', encoding='utf-8').read())

{
  "best_test_acc": 0.9382,
  "history": [
    {
      "epoch": 1,
      "test_acc": 0.9114,
      "test_loss": 0.3124867036938667,
      "seconds": 8.830546855926514
    },
    {
      "epoch": 2,
      "test_acc": 0.93,
      "test_loss": 0.24927375018596648,
      "seconds": 8.388640880584717
    },
    {
      "epoch": 3,
      "test_acc": 0.9382,
      "test_loss": 0.21502381637692453,
      "seconds": 8.359792709350586
    }
  ]
}


## `infer.py` — 13장 infer.py: 저장된 가중치를 불러와 새 입력에 대해 예측한다. 학습 코드는 없다.

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 13장 infer.py: 저장된 가중치를 불러와 새 입력에 대해 예측한다. 학습 코드는 없다.
# 실행 예:
#   python infer.py --run lr005                # results/lr005/ 의 모델로 시험 이미지 8장 예측
#   python infer.py --run lr005 --index 42     # 42번째 시험 이미지 하나
import argparse
import json
import os

import torch
from torchvision import datasets, transforms

from train import MLP                        # 같은 모델 정의를 다시 쓴다 (복사하지 않는다)


def parse_args():
    p = argparse.ArgumentParser(description="MNIST MLP 추론")
    p.add_argument("--results", default="results")
    p.add_argument("--run", required=True, help="불러올 실행 이름 (results/{run}/)")
    p.add_argument("--which", default="best", choices=["best", "last"])
    p.add_argument("--index", type=int, default=None, help="시험 이미지 하나의 번호. 없으면 앞 8장")
    return p.parse_args()


def load_model(out_dir, which):
    with open(os.path.join(out_dir, "config.json"), encoding="utf-8") as f:
        config = json.load(f)                                            # 학습 때의 설정을 읽어
    model = MLP(hidden=config["hidden"])                                 # 같은 모양의 모델을 만들고
    state = torch.load(os.path.join(out_dir, f"model_{which}.pt"), map_location="cpu")
    model.load_state_dict(state)                                         # 가중치를 채운다
    model.eval()                                                         # 추론 모드
    return model, config


def main():
    args = parse_args()
    out_dir = os.path.join(args.results, args.run)
    model, config = load_model(out_dir, args.which)
    print(f"모델 불러옴: {out_dir}/model_{args.which}.pt  (hidden={config['hidden']}, 학습 lr={config['lr']})")

    test_set = datasets.MNIST(config["data_root"], train=False, download=True, transform=transforms.ToTensor())
    indices = [args.index] if args.index is not None else list(range(8))

    with torch.no_grad():                                                # 추론에는 기울기가 필요 없다
        for i in indices:
            image, label = test_set[i]                                   # 4장의 __getitem__: (텐서, 정답)
            scores = model(image.unsqueeze(0))                           # 배치 차원 추가 [1,1,28,28]
            probs = torch.softmax(scores, dim=1)[0]                      # 점수 → 확률
            pred = probs.argmax().item()
            mark = "" if pred == label else "  ← 틀림"
            print(f"시험 이미지 {i:5d}: 정답 {label}  예측 {pred}  확신 {probs[pred]:.3f}{mark}")


if __name__ == "__main__":
    main()
```

In [ ]:
!python infer.py --run lr005

모델 불러옴: results\lr005/model_best.pt  (hidden=128, 학습 lr=0.05)
시험 이미지     0: 정답 7  예측 7  확신 0.994
시험 이미지     1: 정답 2  예측 2  확신 0.966
시험 이미지     2: 정답 1  예측 1  확신 0.978
시험 이미지     3: 정답 0  예측 0  확신 0.999
시험 이미지     4: 정답 4  예측 4  확신 0.908
시험 이미지     5: 정답 1  예측 1  확신 0.988
시험 이미지     6: 정답 4  예측 4  확신 0.946
시험 이미지     7: 정답 9  예측 9  확신 0.965


In [ ]:
!python infer.py --run lr005 --index 8

모델 불러옴: results\lr005/model_best.pt  (hidden=128, 학습 lr=0.05)
시험 이미지     8: 정답 5  예측 6  확신 0.975  ← 틀림


## 다음 단계

- 2단계 「주석 달기」: [ch13_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch13_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch13_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch13_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.